In [1]:
# 第5章｜作業：策略優化：每次開啟 notebook，先執行這一格（安裝套件、登入 FinLab、開啟資料快取）
%pip install -q finlab==2.0.21 ta-lib==0.8.1 lightgbm
import finlab
from finlab import data

finlab.login()
data.set_storage(data.FileStorage())

Note: you may need to restart the kernel to use updated packages.


已登入（使用快取憑證）。


In [2]:
QUICK_RUN = False  # True：減少樹的數量與參數組合，只用來快速檢查程式能不能跑

# 作業：策略優化

**對應影片**：第 5 章 單元 4「作業：策略優化」

**和影片的差異**：資料集與模型的寫法沿用本章單元 1 的新版做法（`finlab.ml.feature`、`finlab.ml.label`、`sim()`）。
另外把資料切成**訓練、驗證、測試**三段，教你用不會自欺欺人的方式優化策略。

這本是**起始 notebook**：從頭執行會完成一輪基本的參數搜尋。你的任務寫在最後的「作業」段落。

In [3]:
import itertools

import numpy as np
import pandas as pd
import finlab.ml.feature as feature
import finlab.ml.label as label
from finlab.backtest import sim
from finlab.dataframe import FinlabDataFrame
from lightgbm import LGBMRegressor

TRAIN_END = '2013-12-31'
VALIDATION_START, VALIDATION_END = '2014-01-01', '2015-12-31'
TEST_START = '2016-01-01'
RANDOM_STATE = 0

## 為什麼要切三段？

| 期間 | 用途 |
| --- | --- |
| 訓練 | 訓練模型 |
| 驗證 | 比較不同的設定（特徵、持股數、模型參數），挑出最好的 |
| 測試 | 設定全部決定之後，**只看一次**，估計策略未來的表現 |

如果一邊看測試期間的結果、一邊調整設定，測試期間就變成另一個訓練期間，回測績效會一直「變好」，實際交易卻不會。

## 1. 資料集（和本章單元 1 相同的特徵）

In [4]:
REVENUE_PERIODS = [1, 3, 6, 12]
REVENUE_SHORT_WINDOW, REVENUE_LONG_WINDOW = 3, 12
PRICE_PERIODS = [5, 10, 20, 60, 120, 240]
LIQUIDITY_WINDOW = 20
MIN_AVG_VOLUME_SHARES = 500_000

revenue = data.get('monthly_revenue:當月營收')
adj_close = data.get('etl:adj_close')


def rsv(price: pd.DataFrame, period: int) -> pd.DataFrame:
    low = price.rolling(period, min_periods=1).min()
    high = price.rolling(period, min_periods=1).max()
    return (price - low) / (high - low)


FEATURE_GROUPS = {
    'revenue': {
        **{f'revenue_growth_{n}m': revenue.pct_change(n) for n in REVENUE_PERIODS},
        'revenue_yoy': data.get('monthly_revenue:去年同月增減(%)'),
        'revenue_momentum': revenue.average(REVENUE_SHORT_WINDOW) / revenue.average(REVENUE_LONG_WINDOW),
    },
    'bias': {f'bias_{n}': adj_close / adj_close.average(n) for n in PRICE_PERIODS},
    'momentum': {f'return_{n}': adj_close.pct_change(n) for n in PRICE_PERIODS},
    'rsv': {f'rsv_{n}': rsv(adj_close, n) for n in PRICE_PERIODS},
}

rebalance_dates = revenue.index
liquid = data.get('price:成交股數').average(LIQUIDITY_WINDOW) > MIN_AVG_VOLUME_SHARES
all_features = {name: frame for group in FEATURE_GROUPS.values() for name, frame in group.items()}

dataset = feature.combine(all_features, resample=rebalance_dates, sample_filter=liquid)
dataset = dataset.replace([np.inf, -np.inf], np.nan)
dataset['return'] = label.return_percentage(dataset.index, resample=rebalance_dates)
dataset['rank'] = dataset['return'].groupby(level='datetime').rank(pct=True)
dates = dataset.index.get_level_values('datetime')
print(f'{len(dataset):,} 筆資料')

151,535 筆資料


## 2. 可調整的策略

`evaluate(config, start, end)` 用訓練資料訓練 LightGBM，在指定期間每月買分數最高的 `n_stocks` 檔並回測。
`config` 可以調整：使用哪些特徵群組、持股數、LightGBM 的葉子數。

In [5]:
N_TREES = 50 if QUICK_RUN else 300
STATS = ['cagr', 'max_drawdown', 'daily_sharpe']


def evaluate(config: dict, start: str, end: str | None = None) -> dict:
    feature_names = [name for group in config['feature_groups'] for name in FEATURE_GROUPS[group]]
    train = dataset[(dates <= TRAIN_END) & dataset['rank'].notna()]

    model = LGBMRegressor(
        n_estimators=N_TREES, learning_rate=0.03, num_leaves=config['num_leaves'], min_child_samples=200,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
    ).fit(train[feature_names], train['rank'])

    period = dataset.loc[start:end] if end else dataset.loc[start:]
    score = pd.Series(model.predict(period[feature_names]), index=period.index)
    position = FinlabDataFrame(score.unstack('instrument')).is_largest(config['n_stocks'])
    stats = sim(position, name='homework').get_stats()
    return {key: stats[key] for key in STATS}


BASELINE = {'feature_groups': list(FEATURE_GROUPS), 'n_stocks': 20, 'num_leaves': 31}
evaluate(BASELINE, VALIDATION_START, VALIDATION_END)

{'cagr': 0.28945060888455054,
 'max_drawdown': -0.2455894462347431,
 'daily_sharpe': 1.3194656028605731}

## 3. 在驗證期間搜尋設定

In [6]:
SEARCH_SPACE = {
    'feature_groups': [list(FEATURE_GROUPS), ['revenue', 'momentum'], ['bias', 'momentum', 'rsv']],
    'n_stocks': [10, 20] if QUICK_RUN else [10, 20, 40],
    'num_leaves': [15] if QUICK_RUN else [15, 63],
}

results = []
for values in itertools.product(*SEARCH_SPACE.values()):
    config = dict(zip(SEARCH_SPACE, values))
    results.append({**config, 'feature_groups': '+'.join(config['feature_groups']),
                    **evaluate(config, VALIDATION_START, VALIDATION_END)})

results = pd.DataFrame(results).sort_values('daily_sharpe', ascending=False, ignore_index=True)
results.style.format({'cagr': '{:.2%}', 'max_drawdown': '{:.2%}', 'daily_sharpe': '{:.2f}'})

,feature_groups,n_stocks,num_leaves,cagr,max_drawdown,daily_sharpe
0,revenue+bias+momentum+rsv,10,15,36.33%,-22.38%,1.59
1,revenue+bias+momentum+rsv,20,15,25.09%,-30.50%,1.23
2,bias+momentum+rsv,20,15,24.13%,-23.51%,1.20
3,revenue+bias+momentum+rsv,10,63,27.25%,-23.61%,1.19
4,bias+momentum+rsv,20,63,24.47%,-23.82%,1.17
5,revenue+bias+momentum+rsv,40,15,22.00%,-25.70%,1.14
6,bias+momentum+rsv,10,15,23.86%,-29.80%,1.12
7,revenue+bias+momentum+rsv,20,63,22.75%,-26.74%,1.08
8,bias+momentum+rsv,10,63,21.60%,-26.88%,0.98
9,bias+momentum+rsv,40,15,16.30%,-28.80%,0.86


## 4. 最後才看測試期間

用驗證期間夏普值最高的設定，在測試期間跑一次：

In [7]:
best = results.iloc[0]
best_config = {
    'feature_groups': best['feature_groups'].split('+'),
    'n_stocks': int(best['n_stocks']),
    'num_leaves': int(best['num_leaves']),
}
print('最佳設定：', best_config)
pd.DataFrame({
    'validation': best[STATS],
    'test': evaluate(best_config, TEST_START),
}).T.style.format({'cagr': '{:.2%}', 'max_drawdown': '{:.2%}', 'daily_sharpe': '{:.2f}'})

最佳設定： {'feature_groups': ['revenue', 'bias', 'momentum', 'rsv'], 'n_stocks': 10, 'num_leaves': 15}


,cagr,max_drawdown,daily_sharpe
validation,36.33%,-22.38%,1.59
test,18.00%,-27.98%,0.87


## 作業

以下每一題都只能用**驗證期間**做決定，全部完成後才看一次測試期間：

1. **新特徵**：在 `FEATURE_GROUPS` 加一個新群組，例如本益比、股價淨值比（`price_earning_ratio:本益比`、`price_earning_ratio:股價淨值比`）或三大法人買賣超（`institutional_investors_trading_summary:投信買賣超股數`）。
2. **換股頻率**：目前每月換股。把 `rebalance_dates` 改成每季一次，會不會降低交易成本、提升報酬？
3. **風險控制**：在 `sim()` 加上 `stop_loss=0.1`（虧損 10% 停損）或 `position_limit` 限制單一股票權重，最大回檔有沒有變小？
4. **集成**：把本章單元 1 的神經網路、隨機森林加回來，和只用 LightGBM 相比如何？

最後寫下你選的設定，以及驗證期間與測試期間的績效差距。差距越小，代表你的優化越可信。